In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
os.chdir(os.path.dirname(os.path.abspath("__file__")))

In [3]:
# ============
# Load Classes
# ============

classes = ['camel', 'crocodile', 'dragon', 'elephant', 'lion', 'panda', 'rhinoceros', 'squirrel', 'tiger', 'zebra']

X_list, y_list = [], []

for i, cls in enumerate(classes):
    data = np.load(f"data/full_numpy_bitmap_{cls}.npy")[:80000]
    labels = np.full(len(data), i)
    X_list.append(data)
    y_list.append(labels)

In [4]:
# ===================
# Create full arrays
# ===================

X = np.concatenate(
    X_list,
    axis=0,
).reshape(-1, 1, 28, 28)
y = np.concatenate(y_list, axis=0,)

In [ ]:
# ===================
# Create DataLoaders
# ===================

X = X / 255.0 # normalize

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X_t, y_t, train_size=0.8, random_state=42)


train_dl = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_dl = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=True)

In [6]:
class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        logits = self.sequential(x)
        return logits

In [7]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = ConvolutionalNetwork().to(device)

learning_rate = 1e-3
batch_size = 32
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [8]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 1000 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss {loss:>7f}\t[{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"test error: \n accuracy: {(100*correct):>0.1f}%, average loss: {test_loss:8>f} \n")

In [9]:
epochs = 5
for t in range(epochs):
    print(f"epoch {t + 1}\n-------------------------------------")
    train_loop(train_dl, model, loss_fn, optimizer)
    test_loop(test_dl, model, loss_fn)
print("Done!")

epoch 1
-------------------------------------
loss 2.311463	[   32/640000]
loss 0.528967	[32032/640000]
loss 0.535848	[64032/640000]
loss 0.552294	[96032/640000]
loss 1.240294	[128032/640000]
loss 0.521374	[160032/640000]
loss 0.855630	[192032/640000]
loss 0.537197	[224032/640000]
loss 0.531468	[256032/640000]
loss 0.759892	[288032/640000]
loss 0.414007	[320032/640000]
loss 0.838545	[352032/640000]
loss 0.507401	[384032/640000]
loss 0.671831	[416032/640000]
loss 0.312797	[448032/640000]
loss 0.670296	[480032/640000]
loss 0.338952	[512032/640000]
loss 0.627863	[544032/640000]
loss 0.433323	[576032/640000]
loss 0.621662	[608032/640000]
test error: 
 accuracy: 84.1%, average loss: 0.490928 

epoch 2
-------------------------------------
loss 0.603997	[   32/640000]
loss 0.619130	[32032/640000]
loss 0.418788	[64032/640000]
loss 0.542810	[96032/640000]
loss 0.385955	[128032/640000]
loss 0.595860	[160032/640000]
loss 0.623031	[192032/640000]
loss 0.198413	[224032/640000]
loss 0.323285	[25603